# 🍔 Swiggy & Zomato Analytics - Customer Sentiment Mining & Text Analytics

Welcome to the **Customer Sentiment Mining & NLP Pipeline**! In this notebook, we implement natural language processing (NLP) techniques on customer review feedback logs to identify why customers are satisfied or disgruntled. 

Using **TextBlob** and **NLTK**, we clean text comments, purge high-frequency semantic stopwords, compute sentiment scores, classify comments into positive/negative classes, plot a high-density word cloud, and extract a complaint leaderboard for logistics managers.

### 🎯 NLP Checklist:
1. **Text Cleansing**: Convert comments to lowercase, strip punctuation, numbers, and trim whitespace.
2. **Stopwords Purge**: Load standard English stopwords via NLTK, append custom food delivery words, and remove them.
3. **Sentiment Polarity**: Extract polarity values ranging from `-1.0` (most negative) to `+1.0` (most positive).
4. **Sentiment Classification**: Segment orders into `Positive`, `Negative`, and `Neutral` buckets.
5. **Word Cloud**: Plot a visual density cloud of top terms using the `wordcloud` library.
6. **Top Complaint Keywords**: Isolate negative feedback and plot a frequency bar chart of key operational failures.

In [ ]:
import pandas as pd
import numpy as np
import re
import os
import matplotlib.pyplot as plt
from textblob import TextBlob
import nltk
from nltk.corpus import stopwords
from wordcloud import WordCloud
from collections import Counter

# Ensure NLTK Stopwords are available
try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords', quiet=True)

print("NLP and visualization libraries loaded successfully!")

## 📥 1. Ingesting Cleaned Dataset
We load our cleaned dataset `data/cleaned_data.csv` produced by our data cleaning notebook.

In [ ]:
cleaned_path = "../data/cleaned_data.csv"
if not os.path.exists(cleaned_path):
    cleaned_path = "data/cleaned_data.csv"

df = pd.read_csv(cleaned_path)
print(f"Ingested dataset shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head(3)

## 🧹 2. Cleaning Text Reviews
Raw text comments are messy, containing punctuation marks, numbers, and capitalizations. We write a function that standardizes the review into lowercase characters and strips special markings.

In [ ]:
def clean_raw_review(text):
    if pd.isnull(text):
        return ""
    # 1. Convert to lowercase
    text = str(text).lower()
    # 2. Remove numbers, special symbols and punctuation
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # 3. Collapse multiple whitespaces into a single space
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_review'] = df['review'].apply(clean_raw_review)
df[['review', 'clean_review']].head(5)

## 🚫 3. Removing Stopwords
Common functional terms like *'the'*, *'and'*, *'is'* do not represent actual customer opinions. We remove them using NLTK's English stop list. We will also include common food delivery neutral terms like *'food'*, *'order'*, *'restaurant'* to prevent them from skewing our analytics.

In [ ]:
# Load NLTK standard English stoplist
try:
    stop_words = set(stopwords.words('english'))
except Exception:
    # Robust fallback standard stop list in case of environment constraints
    stop_words = set(["i", "me", "my", "myself", "we", "our", "ours", "ourselves", "you", "your", "yours", 
                      "he", "him", "his", "she", "her", "it", "its", "they", "them", "their", "what", 
                      "which", "who", "whom", "this", "that", "these", "those", "am", "is", "are", 
                      "was", "were", "be", "been", "being", "have", "has", "had", "having", "do", 
                      "does", "did", "doing", "a", "an", "the", "and", "but", "if", "or", "because", 
                      "as", "until", "while", "of", "at", "by", "for", "with", "about", "against", 
                      "between", "into", "through", "during", "before", "after", "above", "below", 
                      "to", "from", "up", "down", "in", "out", "on", "off", "over", "under", "again", 
                      "further", "then", "once", "here", "there", "when", "where", "why", "how", 
                      "all", "any", "both", "each", "few", "more", "most", "other", "some", "such", 
                      "no", "nor", "not", "only", "own", "same", "so", "than", "too", "very", 
                      "can", "will", "just", "should", "now"])

# Add industry specific neutral words to the stoplist
industry_neutrals = ["food", "order", "ordered", "restaurant", "delivery", "partner", "delivered", "placed"]
stop_words.update(industry_neutrals)

def purge_stopwords(text):
    words = text.split()
    filtered_words = [w for w in words if w not in stop_words]
    return " ".join(filtered_words)

df['clean_review_no_stop'] = df['clean_review'].apply(purge_stopwords)
df[['clean_review', 'clean_review_no_stop']].head(5)

## 🧠 4. Analyzing Sentiment Polarity
We calculate the semantic polarity score of the stopwords-filtered text using TextBlob. The polarity score is a float value within `[-1.0, 1.0]`, where negative represents critical feedback and positive represents praise.

In [ ]:
def calculate_text_polarity(text):
    if not text:
        return 0.0
    return TextBlob(str(text)).sentiment.polarity

df['sentiment_polarity'] = df['clean_review_no_stop'].apply(calculate_text_polarity)
df[['review', 'sentiment_polarity']].head(5)

## 🏷️ 5. Sentiment Classification
We segment polarity values into business categories:
- **Positive**: Polarity > 0.1
- **Negative**: Polarity < -0.1
- **Neutral**: Polarity between -0.1 and 0.1

In [ ]:
def classify_polarity_class(polarity):
    if polarity > 0.1:
        return 'Positive'
    elif polarity < -0.1:
        return 'Negative'
    else:
        return 'Neutral'

df['sentiment_category'] = df['sentiment_polarity'].apply(classify_polarity_class)

print("=== Sentiment Classification Distribution ===")
print(df['sentiment_category'].value_counts())
print("\n=== Percentage Share ===")
print(df['sentiment_category'].value_counts(normalize=True) * 100)

## ☁️ 6. Word Cloud Visualization
A word cloud represents a powerful visual tool for qualitative exploration, scaling words in size relative to their overall frequency count.

In [ ]:
# Combine all cleaned and stopword-purged review strings
full_corpus = " ".join(df['clean_review_no_stop'])

# Create and generate a word cloud image
wordcloud = WordCloud(
    width=1000,
    height=600,
    background_color='#0c0f16', 
    colormap='Oranges', 
    max_words=80
).generate(full_corpus)

# Plot using Matplotlib
plt.figure(figsize=(12, 7), facecolor='#0c0f16')
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title("Zomato & Swiggy Customer Sentiment Word Cloud", color='white', fontsize=18, pad=15)
plt.tight_layout(pad=0)
plt.show()

## 🚨 7. Identifying Top Complaint Keywords
To help regional logistics and restaurant managers address system failures, we isolate only **Negative** customer reviews and plot a frequency bar chart of complaint keywords (e.g. *cold*, *late*, *stale*, *spilled*).

In [ ]:
# Filter for negative feedback comments only
negative_corpus = df[df['sentiment_category'] == 'Negative']['clean_review_no_stop']

# Extract and count individual keywords
negative_words = " ".join(negative_corpus).split()
complaint_counts = Counter(negative_words)

# Retrieve top 12 complaint words
top_complaint_keys = complaint_counts.most_common(12)
complaints_df = pd.DataFrame(top_complaint_keys, columns=['Keyword', 'Frequency'])

# Render horizontal bar plot
plt.figure(figsize=(10, 6), facecolor='#0c0f16')
ax = plt.subplot(111, facecolor='#111823')

plt.barh(complaints_df['Keyword'], complaints_df['Frequency'], color='#e53e3e', edgecolor='#ff4d4d')

# Apply styling
plt.title("Top Customer Complaint Keywords (Operational System Failures)", color='white', fontsize=15, pad=15)
plt.xlabel("Frequency (Occurrences)", color='white', fontsize=12)
plt.ylabel("Complaint Keyword", color='white', fontsize=12)

ax.tick_params(colors='white', labelsize=11)
ax.spines['bottom'].set_color('#2d3748')
ax.spines['left'].set_color('#2d3748')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='x', color='#23354d', linestyle='--', alpha=0.7)

plt.gca().invert_yaxis() # Put highest frequency at the top
plt.tight_layout()
plt.show()

## 💾 8. Save Enriched Dataset
We save the final preprocessed dataframe containing all clean review text and sentiment tags back to `data/cleaned_data.csv` to keep our Streamlit dashboard and SQL tables completely up-to-date.

In [ ]:
export_path = "../data/cleaned_data.csv"
if not os.path.exists(os.path.dirname(export_path)):
    export_path = "data/cleaned_data.csv"

df.to_csv(export_path, index=False)
print(f"Enriched dataset successfully saved. Current shape: {df.shape[0]} rows, {df.shape[1]} columns.")